# 02 - Custom Tavily Search Agent

## Learning objective

Build an agent that can decide when and how to search the internet using a **custom Tavily tool**.

```text
User request
    -> Gemini decides whether a search is needed
    -> Gemini creates a tool call
    -> Python executes the Tavily search function
    -> Search results return to Gemini
    -> Gemini may search again or produce a final answer
```

This notebook separates the workflow into small cells so that the Python syntax, tool definition, agent construction, and returned state can be inspected independently.

> **Quota note:** The component-test cells do not call Gemini unless explicitly stated. The Tavily test cell makes one Tavily request. The final agent cell can make several Gemini and Tavily requests during one execution.


## 1. Check the active Python kernel

The executable should point to the Conda environment being used for notebooks, such as `nlp_env`.


In [1]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)


Python executable: C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\python.exe
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


## 2. Optional package installation

The notebook folder and Python-script folder are siblings. If the course packages are not installed in the active kernel, remove the `#` from the next line and run the cell once.

After installation, use **Kernel -> Restart Kernel** in the Jupyter menu.


In [ ]:
# !pip install -r "../langchain-course/requirements.txt"


## 3. Corporate SSL setup

`truststore` makes Python use the Windows certificate store. Run this before importing packages that create HTTPS clients.


In [2]:
import truststore

truststore.inject_into_ssl()

print("Windows trust store injected successfully.")


Windows trust store injected successfully.


## 4. Load environment variables

The existing `.env` file remains in the sibling `langchain-course` folder. This avoids maintaining duplicate API keys.

The code checks whether the required variables exist without printing their values.


In [3]:
import os
from pathlib import Path

from dotenv import load_dotenv

ENV_PATH = Path("../langchain-course/.env").resolve()

print("Expected .env path:", ENV_PATH)
print(".env file exists:", ENV_PATH.exists())

load_dotenv(dotenv_path=ENV_PATH)

print("Google API key loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("Tavily API key loaded:", bool(os.getenv("TAVILY_API_KEY")))
print("LangSmith API key loaded:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith tracing enabled:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith project:", os.getenv("LANGSMITH_PROJECT"))


Expected .env path: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course\.env
.env file exists: True
Google API key loaded: True
Tavily API key loaded: True
LangSmith API key loaded: True
LangSmith tracing enabled: true
LangSmith project: Langchain Course


## 5. Import the required components

- `create_agent` builds the ready-made LangChain agent loop.
- `tool` converts a normal Python function into a LangChain tool.
- `HumanMessage` represents the user's message.
- `ChatGoogleGenerativeAI` connects LangChain to Gemini.
- `TavilyClient` is Tavily's direct Python SDK.


In [4]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from tavily import TavilyClient

print("Course imports completed successfully.")


Course imports completed successfully.


## 6. Create the direct Tavily client

The client reads `TAVILY_API_KEY` from the environment when it is created.

Creating the client does not perform a search.


In [5]:
tavily = TavilyClient()

print("Tavily client type:", type(tavily))


Tavily client type: <class 'tavily.tavily.TavilyClient'>


## 7. Define a custom LangChain tool

The original Python function accepts one argument:

```text
query: str
```

The `@tool` decorator turns the function into a LangChain tool. LangChain uses:

- The function name as the tool name
- The docstring as the tool description
- The type hint as part of the input schema
- The function body as the executable implementation

The model can request this tool, but the model does not execute Tavily itself. Python executes the function after the agent receives the tool call.


In [6]:
@tool
def search(query: str) -> dict:
    """Search the internet for current information.

    Args:
        query: The search query to send to Tavily.

    Returns:
        The Tavily search-result dictionary.
    """
    print(f"Searching for: {query}")
    return tavily.search(query=query)


print("Custom search tool created.")


Custom search tool created.


## 8. Inspect what `@tool` created

This cell does not call Tavily or Gemini. It inspects the tool metadata available to the model.


In [7]:
print("Tool object type:", type(search))
print("Tool name:", search.name)
print("Tool description:", search.description)
print("Tool argument schema:", search.args)
print("Tool Pydantic schema:", search.args_schema)


Tool object type: <class 'langchain_core.tools.structured.StructuredTool'>
Tool name: search
Tool description: Search the internet for current information.

    Args:
        query: The search query to send to Tavily.

    Returns:
        The Tavily search-result dictionary.
Tool argument schema: {'query': {'title': 'Query', 'type': 'string'}}
Tool Pydantic schema: <class 'langchain_core.utils.pydantic.search'>


## 9. Understand the generated input schema

The model does not see your Python source code. The model receives a structured description indicating that the tool requires a string argument called `query`.

The next cell converts the Pydantic argument schema into a normal dictionary for easier inspection.


In [9]:
tool_schema = search.args_schema.model_json_schema()

tool_schema


{'description': 'Search the internet for current information.\n\nArgs:\n    query: The search query to send to Tavily.\n\nReturns:\n    The Tavily search-result dictionary.',
 'properties': {'query': {'title': 'Query', 'type': 'string'}},
 'required': ['query'],
 'title': 'search',
 'type': 'object'}

## 10. Optional: test the Tavily tool directly

> **This cell makes one Tavily API request but does not call Gemini.**

Direct tool testing separates the external-service behavior from agent behavior. It helps answer: "Does the tool work independently before I give it to the agent?"

Run only when Tavily API access is available.


In [10]:
direct_search_result = search.invoke(
    {"query": "LangChain tool calling documentation"}
)

print("Top-level result keys:", direct_search_result.keys())


Searching for: LangChain tool calling documentation
Top-level result keys: dict_keys(['query', 'follow_up_questions', 'answer', 'images', 'results', 'response_time', 'request_id'])


In [12]:
direct_search_result

{'query': 'LangChain tool calling documentation',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://dev.to/aiengineering/a-beginners-guide-to-tool-calling-in-langchain-l3a',
   'title': 'A Beginner’s Guide to Tool Calling in LangChain - DEV Community',
   'content': 'Tool calling in LangChain follows a simple but powerful pattern. It happens in four main steps: creating the tool, binding it to a model, letting the model decide when to use it, and finally executing the tool.\n\nLet’s walk through each step.\n\n### 1. Tool Creation\n\nThis is where you define the tool you want the model to use. A tool is just a function, but it also needs a name, description, and a schema for its input. LangChain provides a `tool()` helper to make this process easy. [...] Tool calling (also referred to as function calling) is a powerful technique that allows language models to take action by invoking external tools in a structured and predictable way. Instead of 

## 11. Optional: inspect a direct Tavily result

Run this only after executing the previous cell. Inspecting an existing variable does not make another API request.


In [13]:
print("Original query:", direct_search_result.get("query"))
print("Number of results:", len(direct_search_result.get("results", [])))

for index, item in enumerate(direct_search_result.get("results", [])[:3], start=1):
    print(f"Result {index}")
    print("Title:", item.get("title"))
    print("URL:", item.get("url"))


Original query: LangChain tool calling documentation
Number of results: 10
Result 1
Title: A Beginner’s Guide to Tool Calling in LangChain - DEV Community
URL: https://dev.to/aiengineering/a-beginners-guide-to-tool-calling-in-langchain-l3a
Result 2
Title: Tools - Docs by LangChain
URL: https://docs.langchain.com/oss/python/langchain/tools
Result 3
Title: Tool calling - Docs by LangChain
URL: https://docs.langchain.com/oss/python/langchain/frontend/tool-calling


## 12. Initialize Gemini

Creating the model object does not normally send a model request.

`max_retries=2` allows the integration to retry a limited number of transient failures.


In [21]:
MODEL = "gemini-3.5-flash-lite" #"gemini-3.6-flash"

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    max_retries=2,
)

print("Model object type:", type(llm))
print("Configured model:", MODEL)


Model object type: <class 'langchain_google_genai.chat_models.ChatGoogleGenerativeAI'>
Configured model: gemini-3.5-flash-lite


## 13. Create the tools list

An agent can receive multiple tools. This notebook supplies only the custom search tool.


In [22]:
tools = [search]

print("Number of tools:", len(tools))
print("Available tool names:", [current_tool.name for current_tool in tools])


Number of tools: 1
Available tool names: ['search']


## 14. Create the agent

`create_agent()` hides the internal model-tool loop.

Conceptually, the agent performs this cycle:

```text
Call Gemini
-> Check whether Gemini requested a tool
-> Execute the selected tool
-> Add the tool result to message history
-> Call Gemini again
-> Repeat until Gemini returns a final answer
```

The result is a compiled LangGraph-based agent object.


In [23]:
agent = create_agent(
    model=llm,
    tools=tools,
)

print("Agent object type:", type(agent))


Agent object type: <class 'langgraph.graph.state.CompiledStateGraph'>


## 15. Build the user request separately

Creating the message does not call Gemini.

The request asks for current job postings, so the agent should use the search tool instead of relying only on the model's training knowledge.


In [24]:
question = (
    "Search for 3 job postings for an AI engineer "
    "using LangChain in the Bay Area on LinkedIn "
    "and list their details."
)

user_message = HumanMessage(content=question)

print("Message type:", type(user_message))
print("Message content:", user_message.content)


Message type: <class 'langchain_core.messages.human.HumanMessage'>
Message content: Search for 3 job postings for an AI engineer using LangChain in the Bay Area on LinkedIn and list their details.


## 16. Inspect the agent input structure

The agent expects a dictionary whose `messages` value is a list of message objects.

```python
{
    "messages": [HumanMessage(...)]
}
```

This cell constructs the input but does not invoke the agent.


In [25]:
agent_input = {
    "messages": [user_message]
}

print(agent_input)


{'messages': [HumanMessage(content='Search for 3 job postings for an AI engineer using LangChain in the Bay Area on LinkedIn and list their details.', additional_kwargs={}, response_metadata={})]}


## 17. Run the agent

> **This cell can make multiple Gemini requests and multiple Tavily searches. It may also create a LangSmith trace.**

One execution may follow this pattern:

```text
Gemini call 1 -> Search request 1
Tavily call 1 -> Search results 1
Gemini call 2 -> Search request 2 or final answer
...
```

Run this cell only when both Gemini and Tavily API access are available.


In [26]:
result = agent.invoke(agent_input)

print("Agent execution completed.")


Searching for: AI engineer LangChain Bay Area LinkedIn job postings
Searching for: "AI Application Engineer" Docusign San Francisco LinkedIn
Searching for: "AI Engineer" LangChain Palo Alto OR "San Francisco" LinkedIn job
Agent execution completed.


## 18. Inspect the returned agent state

`result` is not merely the final sentence. It contains the accumulated message state from the complete agent execution.

Inspecting the existing result does not make another API request.


In [27]:
print("Result type:", type(result))
print("Result keys:", result.keys())
print("Number of messages:", len(result["messages"]))


Result type: <class 'dict'>
Result keys: dict_keys(['messages'])
Number of messages: 8


## 19. Review every accumulated message

The message list may contain:

- The original `HumanMessage`
- One or more AI messages requesting the search tool
- Tool-result messages containing Tavily output
- The final AI message

This is the history that `create_agent()` managed automatically.


In [28]:
for index, message in enumerate(result["messages"]):
    print("=" * 70)
    print("Message index:", index)
    print("Message type:", type(message).__name__)
    print("Message text/content preview:")

    message_text = getattr(message, "text", None)

    if message_text:
        print(message_text[:1000])
    else:
        print(str(message.content)[:1000])


Message index: 0
Message type: HumanMessage
Message text/content preview:
Search for 3 job postings for an AI engineer using LangChain in the Bay Area on LinkedIn and list their details.
Message index: 1
Message type: AIMessage
Message text/content preview:
[]
Message index: 2
Message type: ToolMessage
Message text/content preview:
{"query": "AI engineer LangChain Bay Area LinkedIn job postings", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.linkedin.com/jobs/langchain-jobs-worldwide", "title": "Langchain Jobs in Worldwide (17,000+ Open Roles) | LinkedIn", "content": "#### LangChain\n\n San Francisco Bay Area  22 hours ago    \n   Senior AI Engineer - Centre of ExcellenceImage 15 ### Senior AI Engineer - Centre of Excellence\n\n#### Columbus\n\n London, England, United Kingdom  Be an early applicant   1 week ago    \n   AI Engineer (LLM/ Chatbot)Image 16 ### AI Engineer (LLM/ Chatbot)\n\n#### Pantheon Lab Limited\n\n Hong Kong, Hong Kong SA

## 20. Print only the final answer

The final message is the last item in the accumulated message list.

Python index `-1` means the last item.


In [30]:
final_message = result["messages"][-1]

print("Final message type:", type(final_message).__name__)
print("Final answer:")
print(final_message.text)


Final message type: AIMessage
Final answer:
Here are 3 AI Engineer job postings requiring or mentioning **LangChain** in the San Francisco Bay Area, curated from LinkedIn and active technology job boards:

### 1. Deployed Engineer (Bay Area)
* **Company:** LangChain
* **Location:** San Francisco, CA (On-site)
* **Job Details:** 
  * **Role focus:** Working hands-on with customers to architect, co-build, and deploy production-grade AI agents and multi-step workflows. Serving as the bridge between engineering, product, and go-to-market.
  * **Requirements / Nice-to-haves:** Experience deploying AI agents in production using frameworks like **LangChain**, **LangGraph**, or similar; experience with LLM evaluation, observability, or guardrails; cloud environment familiarity (AWS, GCP, Azure, containers, Kubernetes).
  * **Compensation:** Annual OTE range of $165,000 – $315,000 USD.

### 2. Senior AI Engineer
* **Company:** Intapp
* **Location:** Palo Alto, CA
* **Job Details:**
  * **Role f

## 21. Optional: inspect tool calls in AI messages

This cell shows which search queries Gemini selected. It reuses the existing result and does not make another API request.


In [31]:
for index, message in enumerate(result["messages"]):
    tool_calls = getattr(message, "tool_calls", None)

    if tool_calls:
        print(f"Message {index} requested {len(tool_calls)} tool call(s):")

        for tool_call in tool_calls:
            print("  Tool name:", tool_call.get("name"))
            print("  Arguments:", tool_call.get("args"))


Message 1 requested 1 tool call(s):
  Tool name: search
  Arguments: {'query': 'AI engineer LangChain Bay Area LinkedIn job postings'}
Message 3 requested 1 tool call(s):
  Tool name: search
  Arguments: {'query': '"AI Application Engineer" Docusign San Francisco LinkedIn'}
Message 5 requested 1 tool call(s):
  Tool name: search
  Arguments: {'query': '"AI Engineer" LangChain Palo Alto OR "San Francisco" LinkedIn job'}


## 22. What changed from the simple chain?

### Script 1: fixed chain

```text
PromptTemplate -> Gemini -> final response
```

The application determines the full path before execution.

### Script 2: agent

```text
User request -> Gemini -> optional tool -> Gemini -> optional tool -> final response
```

The model dynamically decides:

- Whether a search is needed
- Which query to send
- Whether another search is needed
- When enough information exists to answer

The agent therefore introduces a loop, accumulated state, additional latency, and potentially several API calls.


## 23. Key takeaways

1. `TavilyClient` is Tavily's direct SDK client.
2. `@tool` turns a Python function into a LangChain tool.
3. The function name, docstring, and type hints help define the tool interface.
4. The model requests a tool; Python executes the real function.
5. `create_agent()` manages the model-tool loop automatically.
6. The result contains the complete accumulated message history.
7. `result["messages"][-1].text` prints the final readable answer.
8. One agent execution can make multiple Gemini and Tavily calls.
9. Component-level testing helps separate tool problems from agent problems.

## Practice exercises

- Change the job location while keeping the same agent.
- Ask for a different role type.
- Print only the search queries selected by Gemini.
- Modify the tool docstring and inspect how the tool description changes.
- Add `max_results` handling inside the custom function.
- Compare the custom implementation with the built-in `TavilySearch` tool in the next notebook.
